# Psychometric Item Embedding

**Logic:** Generate semantic embeddings for every unique item text via either a
local Ollama model or a hosted API (OpenAI / Google).

**Workflow:** `item_list` parquet -> embed -> item-keyed embedding parquet.

Runs once for each data split (training / holdout / validation).

In [1]:
import os
import sys

import numpy as np
import polars as pl

# Import the embedding-backend functions from the modular helper.
sys.path.append("helper_functions")
from embedding_functions import get_embeddings, get_embeddings_API, get_embeddings_HF

## Configure splits

Which splits to embed: `""` (training), `"holdout_"` or `"validation_"`.

In [2]:
splits = ["", "holdout_", "validation_"]

## Embed each split

For every split: read the item list, generate embeddings, then stack them into a
wide item-keyed matrix and write it to parquet.

In [3]:
for split in splits:

    # ---- Read in Data ----

    item_list = (
        pl.read_parquet(f"../../data/raw/{split}item_list.parquet")
        .unique(subset="item", keep="first", maintain_order=True)
    )

    # ---- Generate Embeddings ----

    # Local Ollama model
    # embeddings, model_safe = get_embeddings(item_list)

    # API alternative: OpenAI or Gemini
    #embeddings, model_safe = get_embeddings_API(item_list, model="text-embedding-ada-002", dims=3072)

    # --- HF models ---
    embeddings, model_safe = get_embeddings_HF(item_list, model="Qwen/Qwen3-Embedding-4B", batch_size=8, quantize=4)
   
    # ---- Save Data ----
    # Stack embedding vectors into a wide matrix, attach the item text, prefix
    # columns with "emb" for downstream feature selectors.
    emb_matrix = np.vstack(list(embeddings.values()) if isinstance(embeddings, dict) else embeddings)
    item_embeddings_df = pl.from_numpy(
        emb_matrix,
        schema=[f"emb{i + 1}" for i in range(emb_matrix.shape[1])],
        orient="row",
    )
    item_embeddings_df.insert_column(0, item_list.get_column("item"))

    os.makedirs(f"../../data/raw/{model_safe}", exist_ok=True)
    item_embeddings_df.write_parquet(
        f"../../data/raw/{model_safe}/{split}embeddings_raw.parquet"
    )

c:\Users\alexm\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0708 12:49:37.415000 44672 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Batches: 100%|██████████| 33/33 [00:06<00:00,  4.74it/s]


In [4]:
print(model_safe)

Qwen-Qwen3-Embedding-4B-4bit
